# Titanic Dataset Exploration

Report notebook for the Titanic preprocessing pipeline.

In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

from pipeline import TitanicPreprocessor, get_titanic_data, split_data

pre = TitanicPreprocessor()

## Section 1: Load Raw Data & Check Missing Values (Before Cleaning)

In [2]:
raw_df = get_titanic_data(str(project_root / "data" / "raw" / "titanic.csv"))

raw_df.info()
print()
print(raw_df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


## Section 2: Title Extraction

In [3]:
df = pre.extract_title(raw_df.copy())
print(df["Title"].value_counts())

Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Mlle          2
Major         2
Col           2
Countess      1
Capt          1
Ms            1
Sir           1
Lady          1
Mme           1
Don           1
Jonkheer      1
Name: count, dtype: int64


## Section 3: Cleaning & Feature Engineering

In [4]:
df = pre.clean(df)
df = pre.feature_engineer(df)

df.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone
0,0,3,male,22.0,1,0,7.2500,S,Mr,2,0
1,1,1,female,38.0,1,0,71.2833,C,Mrs,2,0
2,1,3,female,26.0,0,0,7.9250,S,Miss,1,1
3,1,1,female,35.0,1,0,53.1000,S,Mrs,2,0
4,0,3,male,35.0,0,0,8.0500,S,Mr,1,1


## Section 4: Missing Values After Cleaning

In [5]:
missing_after_cleaning = df.isnull().sum()
print(missing_after_cleaning)
print()
print("Cabin removed:", "Cabin" not in df.columns)
print("Embarked missing values:", missing_after_cleaning["Embarked"])
print("Age missing values:", missing_after_cleaning["Age"])

Survived        0
Pclass          0
Sex             0
Age           177
SibSp           0
Parch           0
Fare            0
Embarked        0
Title           0
FamilySize      0
IsAlone         0
dtype: int64

Cabin removed: True
Embarked missing values: 0
Age missing values: 177


## Section 5: Train-Validation-Test Split

In [6]:
X_train, X_val, X_test = split_data(df)

print("Training Set:", X_train.shape)
print("Validation Set:", X_val.shape)
print("Testing Set:", X_test.shape)

Training Set: (623, 11)
Validation Set: (134, 11)
Testing Set: (134, 11)


## Section 6: Apply Pipeline & Save

In [7]:
X_train_processed = pre.fit_transform(X_train)
X_val_processed = pre.transform(X_val)
X_test_processed = pre.transform(X_test)

print("Processed Training Set:", X_train_processed.shape)
print("Processed Validation Set:", X_val_processed.shape)
print("Processed Testing Set:", X_test_processed.shape)

joblib.dump(pre.preprocessor, project_root / "titanic_preprocessing_pipeline.pkl")
print("Pipeline Saved Successfully")

Processed Training Set: (623, 26)
Processed Validation Set: (134, 26)
Processed Testing Set: (134, 26)
Pipeline Saved Successfully


## Section 7: Results Summary

### Missing Values Handled

In [8]:
before_missing = raw_df.isnull().sum()
after_missing = df.isnull().sum()

missing_values_handled = pd.DataFrame(
    {
        "Before": [
            before_missing["Age"],
            before_missing["Cabin"],
            before_missing["Embarked"],
        ],
        "After": [
            after_missing["Age"],
            "Removed",
            after_missing["Embarked"],
        ],
    },
    index=["Age", "Cabin", "Embarked"],
)

missing_values_handled

,Before,After
Age,177,177
Cabin,687,Removed
Embarked,2,0


### Feature Engineering

| Feature | Purpose |
| --- | --- |
| FamilySize | Combines SibSp and Parch into total family size on board (plus the passenger). |
| IsAlone | Flags passengers traveling without family members. |
| Title | Captures social status and gender cues extracted from the passenger name. |

### Dataset Split

In [9]:
dataset_split = pd.DataFrame(
    {
        "Split": ["Train", "Validation", "Test"],
        "Raw Shape": [X_train.shape, X_val.shape, X_test.shape],
        "Processed Shape": [
            X_train_processed.shape,
            X_val_processed.shape,
            X_test_processed.shape,
        ],
    }
)

dataset_split

,Split,Raw Shape,Processed Shape
0,Train,"(623, 11)","(623, 26)"
1,Validation,"(134, 11)","(134, 26)"
2,Test,"(134, 11)","(134, 26)"


### Completion Checklist

- [x] Data Cleaned
- [x] Features Engineered
- [x] Dataset Split
- [x] Pipeline Saved